# Step 6 — Break it

*Step 6 of the AI in Industry lab*

---

## Read this before you run anything

You will break the system you just built, in three different ways, and work out which part was actually at fault.

**What you should end up understanding:** Most of the time "the AI is wrong", the retrieval was wrong. This is the most useful debugging instinct in the session.

| | |
|---|---|
| **Cost** | 3 API calls - THE IMPORTANT ONE |
| **Needs earlier steps?** | No. This notebook sets itself up. |
| **Safe to re-run?** | Yes. The k=1 versus k=6 comparison is worth running twice. |

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b>Uses about 3 API calls.</b> Re-running cells is fine, it just uses a little more of your free quota each time.</div>

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; This is the most important notebook in the lab. Do not skim it — read both answers in the k=1 versus k=6 cells side by side before moving on.</b></div>

**Now run the Setup cell.** About 30 seconds. It works even if you skipped every earlier step.

In [ ]:
#@title Setup - run this first (about 30 seconds) { display-mode: "form" }
# Fetches the lab files, installs what is needed, reads your API key.
# Identical in every step notebook, so any step works on its own.
import os, sys, pathlib, subprocess

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if not pathlib.Path("rough").exists():
    print("Downloading the lab files ...")
    subprocess.run("git clone --depth 1 --quiet "
                   "https://github.com/coolMukul/rough.git rough", shell=True)
os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing (slow the first time) ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt", shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

if not os.environ.get("LLM_API_KEY"):
    print("\n  NO API KEY. Click the key icon on the left, add a secret named")
    print("  exactly LLM_API_KEY, paste your key from console.groq.com/keys,")
    print("  and turn ON 'Notebook access'. Then run this cell again.")
else:
    print(f"\nReady. Key ending ...{os.environ['LLM_API_KEY'][-4:]}")

You have a working RAG system. Now find out how it fails — **that is the actual job.**

Three failures, in order.

In [ ]:
from labcore import corpus, tfidf, grounded

chunks, texts, _ = corpus()
retrieve = tfidf(texts)
ask = grounded(retrieve)     # the retrieve-then-answer from step 5, in one call

print("ready")

## (a) Ask something that genuinely is not in there

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = "What are the hostel mess timings?"

In [ ]:
print(ask(question, k=4))

It refuses.

**Refusing correctly is a PASSING score**, not a failure. A system that invents an answer about your attendance is worse than useless.

---

## (b) Starve it

Now the important one. Same model, same question. **The only thing that changes is `k`.**

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = ("I have 68% attendance in one course because I was away at "
            "placement drives. What happens to me?")

### With k=1 — one clause only

In [ ]:
print(ask(question, k=1))

### With k=6 — six clauses

In [ ]:
print(ask(question, k=6))

**Read both answers side by side before going on.**

- `k=1` says you are below 75% and in trouble.
- `k=6` says 68% is *within the 10% condonable range*, and placement activity is a listed ground.

One of those would make a student panic for no reason.

**The model was not wrong. It was starved.** At `k=1` it never saw the condonation clause, so it answered correctly from half the picture — confidently, with no hint anything was missing.

> Most of the time "the AI is wrong", **the retrieval was wrong.**

That is the most useful debugging instinct in this whole session.

---

## (c) The right question in the wrong words

In [ ]:
# ===== EDIT ME, then run the cell below =====
rulebook_words = "shortage of attendance condoned"
student_words  = "exemption for missing too many classes"

In [ ]:
print("rulebook wording ->", retrieve(rulebook_words, k=1)[0][0][:85])
print()
print("student wording  ->", retrieve(student_words,  k=1)[0][0][:85])

The second retrieves something irrelevant.

**"exemption" and "condoned" mean the same thing and share no letters.** The rulebook and the student describe the same rule in different words, and word-matching cannot bridge it.

---

## Now change it yourself

**Find the value of k where the answer flips.** Change `k` in the cell below, run it, and narrow it down.

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = ("I have 68% attendance in one course because I was away at "
            "placement drives. What happens to me?")
k = 2                 # try 1, then 2, then 3... where does it change?

In [ ]:
print(ask(question, k=k))

Then write your own question about a rule that **has an exception attached to it** — those are the ones that need two clauses, and they are where this failure lives.

Your regulations are full of them. Step 3 will help you find one.

---

### Done with step 6

Open the next step's notebook. If something here did not work, **do not stop to debug it** — every step sets itself up from scratch, so the next one will still run.